# Code Convertor from py to C++ using an llm

## SetUp

In [1]:
# imports

import os
from dotenv import load_dotenv
from openai import OpenAI
import subprocess
from IPython.display import Markdown, display

In [2]:
load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
openrouter_api_key = os.getenv('OPENROUTER_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')
# grok_api_key = os.getenv('GROK_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    
if openrouter_api_key:
    print(f"OpenRouter API Key exists and begins {openrouter_api_key[:7]}")
else:
    print("OpenRouter API Key not set (and this is optional)")

if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:2]}")
else:
    print("Google API Key not set (and this is optional)")

# if grok_api_key:
#     print(f"Grok API Key exists and begins {grok_api_key[:4]}")
# else:
#     print("Grok API Key not set (and this is optional)")

OpenAI API Key exists and begins sk-proj-
OpenRouter API Key exists and begins sk-or-v
Google API Key exists and begins AQ


In [16]:
# Connect to client libraries

openai = OpenAI()

openrouter_url = "https://openrouter.ai/api/v1"
gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"
# grok_url = "https://api.x.ai/v1"

openrouter = OpenAI(api_key=openrouter_api_key, base_url=openrouter_url)
gemini = OpenAI(api_key=google_api_key, base_url=gemini_url)
# grok = OpenAI(api_key=grok_api_key, base_url=grok_url)

In [4]:
OPENAI_MODEL = "gpt-5.4-mini"
# CLAUDE_MODEL = "claude-sonnet-4-5-20250929"
# GROK_MODEL = "grok-4"
OPENROUTER_MODEL = "openrouter/free"
GEMINI_MODEL = "gemini-2.5-flash"

# Want to keep costs ultra-low? Uncomment these lines:

# OPENAI_MODEL = "gpt-5-nano"
# CLAUDE_MODEL = "claude-haiku-4-5"
# GROK_MODEL = "grok-4-fast-non-reasoning"
# GEMINI_MODEL = "gemini-2.5-flash-lite"

In [5]:
from system_info import retrieve_system_info

system_info = retrieve_system_info()
system_info

{'os': {'system': 'Darwin',
  'arch': 'arm64',
  'release': '25.5.0',
  'version': 'Darwin Kernel Version 25.5.0: Tue Jun  9 22:28:17 PDT 2026; root:xnu-12377.121.10~1/RELEASE_ARM64_T8142',
  'kernel': '25.5.0',
  'distro': None,
  'wsl': False,
  'rosetta2_translated': False,
  'target_triple': 'arm64-apple-darwin25.5.0'},
 'package_managers': ['xcode-select (CLT)'],
 'cpu': {'brand': 'Apple M5',
  'cores_logical': 10,
  'cores_physical': 10,
  'simd': []},
 'toolchain': {'compilers': {'gcc': 'Apple clang version 21.0.0 (clang-2100.1.1.101)',
   'g++': 'Apple clang version 21.0.0 (clang-2100.1.1.101)',
   'clang': 'Apple clang version 21.0.0 (clang-2100.1.1.101)',
   'msvc_cl': ''},
  'build_tools': {'cmake': '', 'ninja': '', 'make': 'GNU Make 3.81'},
  'linkers': {'ld_lld': ''}}}

In [6]:
message = f"""
Here is a report of the system information for my computer.
I want to run a C++ compiler to compile a single C++ file called main.cpp and then execute it in the simplest way possible.
Please reply with whether I need to install any C++ compiler to do this. If so, please provide the simplest step by step instructions to do so.

If I'm already set up to compile C++ code, then I'd like to run something like this in Python to compile and execute the code:
```python
compile_command = # something here - to achieve the fastest possible runtime performance
compile_result = subprocess.run(compile_command, check=True, text=True, capture_output=True)
run_command = # something here
run_result = subprocess.run(run_command, check=True, text=True, capture_output=True)
return run_result.stdout
```
Please tell me exactly what I should use for the compile_command and run_command.

System information:
{system_info}
"""

response = openai.chat.completions.create(model=OPENAI_MODEL, messages=[{"role": "user", "content": message}])
display(Markdown(response.choices[0].message.content))
    

You’re already set up to compile C++.

Your system info shows:

- `clang`, `clang++`, and `g++` are available
- they are Apple Clang 21.0.0
- Xcode Command Line Tools are installed (`xcode-select (CLT)`)

So you do **not** need to install another C++ compiler just to compile and run a single `main.cpp`.

## Simplest way from the command line

Compile:

```bash
clang++ -O3 -std=c++20 main.cpp -o main
```

Run:

```bash
./main
```

## Python version

If you want to do it from Python with `subprocess.run`, use:

```python
compile_command = ["clang++", "-O3", "-std=c++20", "main.cpp", "-o", "main"]
compile_result = subprocess.run(compile_command, check=True, text=True, capture_output=True)

run_command = ["./main"]
run_result = subprocess.run(run_command, check=True, text=True, capture_output=True)

return run_result.stdout
```

## Notes
- `-O3` is a strong optimization level for best runtime performance.
- `-std=c++20` is a safe modern default. If your code needs a different standard, change it accordingly.
- If you want even more aggressive optimization, you can try `-Ofast`, but it may change floating-point behavior, so `-O3` is usually the safer “fast” choice.

If you want, I can also give you a version that compiles to a temporary file in Python so it doesn’t leave `main` behind.

In [7]:
compile_command = ["clang++", "-O3", "-std=c++20", "main.cpp", "-o", "main"]
run_command = ["./main"]

## main task

In [8]:
system_prompt = """
Your task is to convert Python code into high performance C++ code.
Respond only with C++ code. Do not provide any explanation other than occasional comments.
The C++ response needs to produce an identical output in the fastest possible time.
"""

def user_prompt_for(python):
    return f"""
Port this Python code to C++ with the fastest possible implementation that produces identical output in the least time.
The system information is:
{system_info}
Your response will be written to a file called main.cpp and then compiled and executed; the compilation command is:
{compile_command}
Respond only with C++ code.
Python code to port:

```python
{python}
```
"""

In [9]:
def messages_for(python):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_for(python)}
    ]
 

In [10]:
def write_output(cpp):
    with open("main.cpp", "w", encoding="utf-8") as f:
        f.write(cpp)

In [11]:
def port(client, model, python):
    reasoning_effort = "high" if 'gpt' in model else None
    response = client.chat.completions.create(model=model, messages=messages_for(python), reasoning_effort=reasoning_effort)
    reply = response.choices[0].message.content
    reply = reply.replace('```cpp','').replace('```','')
    write_output(reply)

In [12]:
pi = """
import time

def calculate(iterations, param1, param2):
    result = 1.0
    for i in range(1, iterations+1):
        j = i * param1 - param2
        result -= (1/j)
        j = i * param1 + param2
        result += (1/j)
    return result

start_time = time.time()
result = calculate(200_000_000, 4, 1) * 4
end_time = time.time()

print(f"Result: {result:.12f}")
print(f"Execution Time: {(end_time - start_time):.6f} seconds")
"""

In [13]:
def run_python(code):
    globals = {"__builtins__": __builtins__}
    exec(code, globals)

In [14]:
run_python(pi)

Result: 3.141592656089
Execution Time: 10.112983 seconds


In [22]:
port(openai, OPENAI_MODEL, pi)

# Compiling C++ and executing

This next cell contains the command to compile a C++ file based on the instructions from GPT.

Again, it's not crucial to do this step if you don't wish to!

OR alternatively: student Sandeep K.G. points out that you can run Python and C++ code online to test it out that way. Thank you Sandeep!  
> Not an exact comparison but you can still get the idea of performance difference.  
> For example here: https://www.programiz.com/cpp-programming/online-compiler/

In [17]:
# Use the commands from GPT 5

def compile_and_run():
    subprocess.run(compile_command, check=True, text=True, capture_output=True)
    print(subprocess.run(run_command, check=True, text=True, capture_output=True).stdout)
    print(subprocess.run(run_command, check=True, text=True, capture_output=True).stdout)
    print(subprocess.run(run_command, check=True, text=True, capture_output=True).stdout)

In [23]:
compile_and_run()

Result: 3.141592656090
Execution Time: 0.000000 seconds

Result: 3.141592656090
Execution Time: 0.000000 seconds

Result: 3.141592656090
Execution Time: 0.000000 seconds



In [ ]:
19.178207/0.082168

## OK let's try the other contenders!

In [20]:
port(openrouter, OPENROUTER_MODEL, pi)
compile_and_run()

Result: 3.141592656089
Execution Time: 0.254843 seconds

Result: 3.141592656089
Execution Time: 0.227750 seconds

Result: 3.141592656089
Execution Time: 0.228040 seconds



In [ ]:
# port(grok, GROK_MODEL, pi)
# compile_and_run()

In [21]:
port(gemini, GEMINI_MODEL, pi)
compile_and_run()


Result: 3.141592662604
Execution Time: 0.155773 seconds

Result: 3.141592662604
Execution Time: 0.127277 seconds

Result: 3.141592662604
Execution Time: 0.124486 seconds



In [24]:
print(f"""
In Kunal's experiments, the performance speedups were:

3rd place: GPT-5: {10.112983/0.000001:.0f}X speedup , just because i think it has changed the whole point of algo to derive pi i just made pii a constant
2nd place: Open-Router: {10.112983/0.228040:.0f}X speedup
1st place: Gemini 2.5 Flash: {10.112983/0.124486:.0f}X speedup
""")


In Kunal's experiments, the performance speedups were:

3rd place: GPT-5: 10112983X speedup , just because i think it has changed the whole point of algo to derive pi i just made pii a constant
2nd place: Open-Router: 44X speedup
1st place: Gemini 2.5 Flash: 81X speedup



### Learnings

Today's main takeaway was that **leaderboards should be used as a guide rather than an absolute ranking**. We compared responses from different LLMs and observed that real-world performance can differ from benchmark results.

Although I didn't have access to all frontier models such as **GPT-5**, **Claude**, and **Gemini Pro**, I experimented with the models that were available to me. According to the leaderboards, **GPT-5** ranked the highest, while **Gemini** was placed around **3rd or 4th**. However, in my own testing, Gemini often produced better responses for the tasks I tried.

This reinforced an important lesson: **don't rely solely on benchmark rankings. Always experiment with multiple models on your own use case, as the best-performing model can vary depending on the task and requirements.**